# Trees and Ensembles, Notebook 4
## Ensembles: bagging, random forest, AdaBoost and gradient boosting

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

A single tree is unstable: change a few rows and the root changes (we saw it in Notebook 3).
**Ensembles** turn that into an advantage: instead of one tree, many imperfect trees that vote
or that are added up. There are two families, and it helps to keep them separate from the start:

| Family | How it trains | What it reduces | Examples |
|---|---|---|---|
| **Bagging** | Many trees **in parallel**, each on a randomly drawn copy of the data; at the end they vote | The **variance** (the instability) | Bagging (Breiman, 1996), Random forest (Breiman, 2001) |
| **Boosting** | Many small trees **in sequence**, each focused on what the previous ones got wrong; at the end they are added up | The **bias** (what a small tree cannot capture) | AdaBoost (Freund and Schapire, 1996), Gradient boosting (Friedman, 2001), XGBoost, LightGBM, CatBoost |

Today we build each one **by hand** with the 219 firms, to see the mechanism, and then we
let them loose with `scikit-learn` on 30,000 credit card clients, where you can see what they
are good for.

### What you will learn today

1. Bagging: bootstrap, voting and the "out-of-bag" error.
2. Random forest: the second random draw, and variable importance.
3. AdaBoost: reweighting the difficult firms, round by round.
4. Gradient boosting: adding up staircases that correct the residual, and the XGBoost / LightGBM / CatBoost family.
5. When an ensemble beats a tree, and when it does not.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import binom

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Ready.")


## 1. The data, once more

The same 219 firms and the same partition as in Notebooks 2 and 3. The reference point is
the depth-2 tree: 76.7% on training and 82.2% on test.


| Ratio | What it measures |
|---|---|
| `deuda_activos` | total debt / total assets (leverage) |
| `razon_corriente` | current assets / current liabilities (liquidity) |
| `ventas_deuda` | sales / total debt (ability to service the debt) |
| `ln_activos` | logarithm of total assets (size) |
| `roa` | net income / total assets (profitability) |
| `impago` | **what we want to predict:** 1 if the firm defaulted, 0 if not |


In [ ]:

# The 219 firms with financial ratios (the course default table, with 5 of its ratios).
# It is pasted right here so the notebook does not depend on any external file.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Firms:", len(impago), "  Fraction in default:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# The same partition in every notebook of the set: 146 firms for training, 73 for testing
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Training:", len(X_train), "  Test:", len(X_test))
print("Fraction in default: training", round(y_train.mean(), 3), " test", round(y_test.mean(), 3))


In [ ]:

def evaluar(modelo, nombre):
    acc_tr = accuracy_score(y_train, modelo.predict(X_train)); acc_te = accuracy_score(y_test, modelo.predict(X_test))
    print(f"{nombre:44s} training {acc_tr:.3f}   test {acc_te:.3f}")
    return acc_te

arbol2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
arbol_full = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
evaluar(arbol2, "depth-2 tree (reference)")
evaluar(arbol_full, "unrestricted tree (28 leaves)");


## 2. Bagging by hand: draw, train, vote

**Bootstrap** means drawing $n$ rows **with replacement** from a table of $n$ rows: some come out
two or three times, others not at all. On average 37% ($1/e$) of the rows are left out. Each
draw gives a similar but different table; a tree trained on each one gives similar but different
trees; and when they vote, the errors they do not share cancel out.

**Why it works.** The variance of the average of $B$ correlated trees is

$$
\text{Var} = \rho \, \sigma^2 + \frac{1 - \rho}{B} \, \sigma^2
$$

where $\sigma^2$ is the variance of one tree and $\rho$ the correlation between two trees. With
large $B$ what remains is $\rho \sigma^2$: averaging many trees removes the part they do not share, but not
the part they do share. Lowering $\rho$ is what the random forest does.


In [ ]:

rng = np.random.default_rng(11)
B = 7
n = len(X_train)
muestras = [rng.integers(0, n, n) for _ in range(B)]       # indices drawn with replacement

cuentas = pd.DataFrame({f"sample {b+1}": np.bincount(m, minlength=n) for b, m in enumerate(muestras)})
print("How many times each firm appears in each sample (first 10 firms):")
print(cuentas.head(10).T)
print()
print("Firms left out of each sample:", [(c == 0).sum() for _, c in cuentas.items()], "  (out of 146; on average 1/e = 37%)")


In [ ]:

arboles_bag = [DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train.iloc[m], y_train.iloc[m]) for m in muestras]
for b, a in enumerate(arboles_bag):
    print(f"tree {b+1}: root {RATIOS[a.tree_.feature[0]]} ≤ {a.tree_.threshold[0]:.3f}   test accuracy {accuracy_score(y_test, a.predict(X_test)):.3f}")

votos = np.column_stack([a.predict(X_test) for a in arboles_bag])
mayoria = (votos.mean(axis=1) > 0.5).astype(int)
print()
print(f"Typical tree (average of the 7) on test: {np.mean([accuracy_score(y_test, a.predict(X_test)) for a in arboles_bag]):.3f}")
print(f"Vote of the 7 on test:                  {accuracy_score(y_test, mayoria):.3f}")


Seven samples, seven trees with different roots (there is the instability), and the vote
beats the typical tree of the set. Look at one test firm in detail:


In [ ]:

i = 0
print("Test firm 1:", X_test.iloc[i].round(3).to_dict(), "  actual:", "default" if y_test.iloc[i] else "pays")
print("Votes of the 7 trees:", ["default" if v else "pays" for v in votos[i]], "->", "default" if mayoria[i] else "pays")


### The out-of-bag (OOB) error

Bagging comes with a gift: each tree left out 37% of the firms, and those firms are
data that **this tree never saw**. For each training firm, only the trees that did not use it
are allowed to vote, and the accuracy is measured. It is a free test estimate, without spending
the 73 reserved firms.


In [ ]:

aciertos = []
for j in range(n):
    fuera = [b for b in range(B) if cuentas.iloc[j, b] == 0]
    if not fuera: continue
    voto = np.mean([arboles_bag[b].predict(X_train.iloc[[j]])[0] for b in fuera]) > 0.5
    aciertos.append(int(voto) == y_train.iloc[j])
print(f"Out-of-bag accuracy: {np.mean(aciertos):.3f} over {len(aciertos)} firms   (test accuracy: {accuracy_score(y_test, mayoria):.3f})")


## 3. Random forest: the second random draw

If all the trees have the same root, they look too much alike ($\rho$ high) and voting helps
little. Random forest adds a second random draw: at **each split**, each tree may only look at
a randomly chosen subset of the variables (`max_features`, typically the square root of the
total). This forces the trees to use different variables and to look less alike.


In [ ]:

bag100 = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=100, oob_score=True, random_state=0).fit(X_train, y_train)
rf100 = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=0).fit(X_train, y_train)
rf100_d3 = RandomForestClassifier(n_estimators=100, max_depth=3, max_features=2, oob_score=True, random_state=0).fit(X_train, y_train)
evaluar(arbol2, "depth-2 tree (reference)")
evaluar(bag100, f"bagging, 100 trees (OOB {bag100.oob_score_:.3f})")
evaluar(rf100, f"random forest, 100 trees (OOB {rf100.oob_score_:.3f})")
evaluar(rf100_d3, f"random forest, 100 trees, depth 3, 2 vars. (OOB {rf100_d3.oob_score_:.3f})");


**An honest reading.** With 219 firms and 5 ratios, neither bagging nor random forest beats
the two-level tree on test. A well-chosen small tree already captures almost everything there is
in these data, and with 73 test firms the differences amount to one or two firms. Ensembles
shine with more data, more variables and more noise; we will see that in section 6.
What you see here is the mechanism, not its best case.

### Variable importance

A forest can no longer be read like a rulebook. In exchange it delivers an **importance** per variable:
how much impurity each one reduced, summed over all the splits of all the trees.


In [ ]:

imp = pd.Series(rf100.feature_importances_, index=RATIOS).sort_values(ascending=False)
fig = px.bar(imp, orientation="h", title="Random forest variable importance (accumulated Gini reduction)", labels={"value": "importance", "index": ""})
fig.show()
imp.round(3)


### Why voting works: the binomial calculation

If each classifier errs with probability $\varepsilon$ and the errors are
independent, the majority of $n$ classifiers errs with binomial probability:

$$
P(\text{majority errs}) = \sum_{k > n/2} \binom{n}{k} \varepsilon^k (1 - \varepsilon)^{n - k}
$$

With $\varepsilon = 0.35$ and 25 classifiers, the majority fails 6% of the time. The strong
assumption is independence: in practice the trees look alike ($\rho > 0$) and the improvement is
smaller. That is why the second random draw matters.


In [ ]:

filas = []
for eps in [0.2, 0.35, 0.45, 0.5, 0.6]:
    fila = {"ε (error of each one)": eps}
    for n_clf in [1, 5, 25, 101]:
        fila[f"{n_clf} classifiers"] = binom.sf(n_clf // 2, n_clf, eps)   # P(X > n/2)
    filas.append(fila)
pd.DataFrame(filas).round(3)


With $\varepsilon < 0.5$ the majority's error falls toward zero as classifiers are added; with
$\varepsilon = 0.5$ voting changes nothing; with $\varepsilon > 0.5$ voting makes things **worse**.

## 4. AdaBoost by hand: focusing on the hard cases

Boosting trains in sequence. The first model looks at all firms equally; the second gives
more weight to those the first one misclassified; and so on. At the end they vote with weights: the models that
got more right weigh more. The base models are **stumps**: trees with a single split.

$$
\varepsilon_m = \sum_i w_i \, [h_m(x_i) \neq y_i]
\qquad
\alpha_m = \tfrac{1}{2} \ln \frac{1 - \varepsilon_m}{\varepsilon_m}
\qquad
w_i \leftarrow \frac{w_i \, e^{-\alpha_m y_i h_m(x_i)}}{Z}
\qquad
F(x) = \text{sign}\Big(\sum_m \alpha_m h_m(x)\Big)
$$

$w_i$ is the weight of firm $i$ (they start equal, $1/n$); $h_m$ the stump of round $m$
($+1$ default, $-1$ pays); $\varepsilon_m$ its **weighted** error; $\alpha_m$ its weight in the vote
(large if $\varepsilon$ is small, zero if $\varepsilon = 0.5$); $Z$ normalizes the weights so that
they sum to 1. The misclassified firms ($y \cdot h = -1$) go up in weight; the correctly classified ones go down.
A nice property: after reweighting, the firms that the stump just chosen misclassified
weigh exactly half of the total, so that stump is "neutralized" and the next one
has to find something else.


In [ ]:

y_pm = np.where(y_train == 1, 1, -1); y_pm_test = np.where(y_test == 1, 1, -1)
w = np.ones(n) / n
F_train = np.zeros(n); F_test = np.zeros(len(X_test))
for m in range(5):
    tocon = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_pm, sample_weight=w)
    h = tocon.predict(X_train); h_test = tocon.predict(X_test)
    eps = w[h != y_pm].sum()
    alfa = 0.5 * np.log((1 - eps) / eps)
    F_train += alfa * h; F_test += alfa * h_test
    w = w * np.exp(-alfa * y_pm * h); w = w / w.sum()
    print(f"round {m+1}: stump {RATIOS[tocon.tree_.feature[0]]:16s} ≤ {tocon.tree_.threshold[0]:7.3f}   ε = {eps:.3f}   α = {alfa:.3f}   "
          f"weight of the misclassified afterwards: {w[h != y_pm].sum():.3f}   cumulative: training {np.mean(np.sign(F_train) == y_pm):.3f}  test {np.mean(np.sign(F_test) == y_pm_test):.3f}")


Each stump is only slightly better than chance (look at how $\varepsilon$ rises round by round: each
one works on harder firms), and in each round the misclassified firms end up weighing
exactly half. With five rounds the weighted vote still does not beat the two-level
tree and on test it oscillates: boosting needs more rounds to show what it can do. With
`scikit-learn` and more stumps:


In [ ]:

for rondas in [5, 20, 100]:
    ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=rondas, learning_rate=1.0, random_state=0).fit(X_train, y_train)
    evaluar(ada, f"AdaBoost, {rondas} stumps")


**Why it works and when it fails.** Boosting reduces bias: the sum of stumps can
represent shapes that no single stump can reach. The symmetric risk: it keeps lowering the training
error even when test no longer improves, and it is sensitive to wrong labels, to which it
gives more and more weight.

## 5. Gradient boosting by hand: adding up staircases that correct the residual

The generalization of AdaBoost that dominates tabular data. The recipe, for predicting a
number:

$$
F_0 = \text{mean of } y
\qquad
r_i = y_i - F_{m-1}(x_i)
\qquad
h_m = \text{small tree that best explains } r
\qquad
F_m = F_{m-1} + \eta \cdot h_m
$$

$r_i$ is the **residual**, what is still missing; with the squared loss it is exactly the negative
gradient of the loss, hence the name. $\eta$ is the **learning rate**: with 1 the whole
tree is added (fast, it overfits); with 0.1 you move a little at a time (more rounds, smoother).

We predict profitability (`roa`) from sales/debt, with a single variable so we can
draw the prediction: a staircase that is refined round by round.


In [ ]:

x_gb = X_train[["ventas_deuda"]]; y_gb = impago.loc[X_train.index, "roa"].values
x_gb_test = X_test[["ventas_deuda"]]; y_gb_test = impago.loc[X_test.index, "roa"].values
eta = 0.5
F = np.full(len(y_gb), y_gb.mean()); F_t = np.full(len(y_gb_test), y_gb.mean())
grilla = pd.DataFrame({"ventas_deuda": np.linspace(0, 8, 400)}); F_g = np.full(len(grilla), y_gb.mean())
fig = go.Figure(); fig.add_trace(go.Scatter(x=x_gb.ventas_deuda, y=y_gb, mode="markers", name="training", opacity=0.4))
print(f"round 0: F = mean = {y_gb.mean():.4f}   training MSE {np.mean((y_gb - F) ** 2):.5f}   test MSE {np.mean((y_gb_test - F_t) ** 2):.5f}")
for m in range(1, 7):
    residuo = y_gb - F
    tocon = DecisionTreeRegressor(max_depth=1, random_state=0).fit(x_gb, residuo)
    F = F + eta * tocon.predict(x_gb); F_t = F_t + eta * tocon.predict(x_gb_test); F_g = F_g + eta * tocon.predict(grilla)
    print(f"round {m}: split sales/debt ≤ {tocon.tree_.threshold[0]:.3f}   step left {tocon.tree_.value[1][0][0]:8.4f}  right {tocon.tree_.value[2][0][0]:8.4f}   "
          f"training MSE {np.mean((y_gb - F) ** 2):.5f}   test MSE {np.mean((y_gb_test - F_t) ** 2):.5f}")
    if m in (1, 3, 6):
        fig.add_trace(go.Scatter(x=grilla.ventas_deuda, y=F_g, name=f"F after {m} rounds", line=dict(width=3 if m == 6 else 1.5)))
fig.update_layout(title="Gradient boosting: the staircase is built round by round (η = 0.5)", xaxis_title="sales / debt", yaxis_title="roa", xaxis_range=[0, 8])
fig.show()


Each round lowers the training error; the test error falls almost always here, but it would
not do so forever: that is why the number of rounds is a parameter, and it is chosen with early
stopping on validation data. Notice also that the second split, in round 3, lands at
sales/debt ≤ 0.027: two firms with almost no sales and very negative roa, to which the
boosting devotes an entire round. That is its way of overfitting.

**The family.** Gradient boosting is the recipe above with any differentiable loss (for
classification, the logistic one: the residual becomes $y - p$). **XGBoost** (2014) adds
explicit regularization on the leaves, handling of missing values and parallelism; **LightGBM** (2017)
grows the trees leaf by leaf and groups values into bins, much faster with millions of
rows; **CatBoost** (2018) handles categorical variables without preprocessing and uses symmetric trees. All
are this same recipe with engineering improvements. `HistGradientBoostingClassifier` from
`scikit-learn` is the "LightGBM-style" version that comes built in.

**The knobs, in order.** Number of rounds and learning rate (they go together); tree
size (stumps for simple effects, depth 3 to 6 for interactions); subsampling of
rows and columns; regularization of the leaves.


In [ ]:

for lr, rondas in [(1.0, 50), (0.1, 50), (0.1, 300), (0.05, 300)]:
    gb = GradientBoostingClassifier(n_estimators=rondas, learning_rate=lr, max_depth=2, random_state=0).fit(X_train, y_train)
    evaluar(gb, f"gradient boosting, η = {lr}, {rondas} rounds, depth 2")


## 6. Where ensembles shine: 30,000 credit card clients

With 219 firms everything ties. Let us change scale: the Taiwan credit card dataset
(Yeh and Lien, 2009), 30,000 clients, 23 variables (limit, age, and for each of the last
six months the payment delay, the bill and the payment) and whether the client defaulted the following month
(22%). Since the clients are imbalanced, we measure with **AUC** (the probability that the
model assigns more risk to a client who did not pay than to one who did; 0.5 is chance, 1 is
perfect), in addition to accuracy.

If you have the link to the course file (`taiwan_credit.csv`), paste it into `URL_TAIWAN`; if not,
it is downloaded from OpenML.


In [ ]:

from sklearn.datasets import fetch_openml

def normalizar_link(url):
    # Converts a Drive or Dropbox "share" link into a direct download link
    import re
    m = re.search(r"drive\.google\.com/file/d/([^/]+)", url)
    if m:
        return "https://drive.google.com/uc?export=download&id=" + m.group(1)
    if "dropbox.com" in url:
        url = url.replace("www.dropbox.com", "dl.dropboxusercontent.com").replace("&dl=0", "").replace("?dl=0", "")
    return url

URL_TAIWAN = ""    # <-- paste here the link to the course CSV (taiwan_credit.csv) if you have it

if URL_TAIWAN != "":
    taiwan_df = pd.read_csv(normalizar_link(URL_TAIWAN))
    X_t = taiwan_df.drop(columns="target").apply(pd.to_numeric); y_t = taiwan_df["target"].astype(int)
    print("Taiwan read from the course file")
else:
    taiwan = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X_t = taiwan.data.apply(pd.to_numeric); y_t = taiwan.target.astype(int)
    print("Taiwan downloaded from OpenML")

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_t.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
               + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_t, y_t, test_size=9000, random_state=0, stratify=y_t)
print("Clients:", len(X_t), "  Variables:", X_t.shape[1], "  Fraction that did not pay:", round(y_t.mean(), 3))
print("Training:", len(Xt_train), "  Test:", len(Xt_test))


In [ ]:

modelos = {
    "árbol de profundidad 2":                 DecisionTreeClassifier(max_depth=2, random_state=0),
    "árbol de profundidad 5":                 DecisionTreeClassifier(max_depth=5, random_state=0),
    "árbol sin freno":                        DecisionTreeClassifier(random_state=0),
    "random forest, 200 árboles":             RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "AdaBoost, 200 tocones":                  AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=0),
    "gradient boosting (hist), 300 rondas":   HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=0),
}
filas = []
for nombre, m in modelos.items():
    t0 = time.time(); m.fit(Xt_train, yt_train); seg = time.time() - t0
    p = m.predict_proba(Xt_test)[:, 1]
    filas.append((nombre, accuracy_score(yt_test, m.predict(Xt_test)), roc_auc_score(yt_test, p), seg))
resultados = pd.DataFrame(filas, columns=["model", "test accuracy", "test AUC", "seconds"]).set_index("model")
print(f"Always predicting 'pays' gets {1 - yt_test.mean():.3f} right")
resultados.round(3)


**Now we are talking.** First, accuracy is useless for comparing: almost all the models give 82%,
which is what predicting "pays" for everyone gives, because only 22% do not pay. You have to look at the AUC.
There the unrestricted tree sinks (it memorizes), the two-level tree falls short (0.69), the
depth-5 one improves (0.75), and the random forest and the gradient boosting win (0.77 and
0.78) by using the 21,000 clients to discover structure that a small tree cannot reach. The
general rule: an ensemble wins in proportion to how much structure there is to discover and to
how much data there is to discover it with. With 219 firms there was not enough of either
thing.

And what is lost: there is no longer a rulebook to read. It is explained with importances and with
examples, and in credit that has a regulatory cost. It is a real trade-off, not a detail.


In [ ]:

rf_t = modelos["random forest, 200 árboles"]
imp_t = pd.Series(rf_t.feature_importances_, index=X_t.columns).sort_values(ascending=False).head(10)
fig = px.bar(imp_t, orientation="h", title="The 10 most important variables according to the random forest (Taiwan)", labels={"value": "importance", "index": ""})
fig.show()


## 7. What to take away

- **Bagging** averages trees trained on randomly drawn copies: it reduces variance. **Random
  forest** adds the random draw of variables so that the trees look less alike, and gives away the
  out-of-bag error and variable importance for free.
- **Boosting** trains in sequence, each model focused on the previous one's error: it reduces
  bias. AdaBoost reweights firms; gradient boosting fits the residual. XGBoost, LightGBM and
  CatBoost are gradient boosting with engineering improvements.
- With few data and few variables, a small tree ties with everything. With lots of data, the
  ensembles win and the unrestricted tree sinks.
- What to ask when someone presents an ensemble: how many rounds or trees and how the
  number was chosen, what error it gives on data it never saw, which variables carry weight and whether they make
  business sense, and how an individual prediction is explained.

## Exercises

**Exercise 1.** In section 2, repeat the bagging by hand with `B = 25` instead of 7. Does the
test accuracy of the vote go up? And the out-of-bag one? How many firms are left without
any tree that has not seen them?

**Exercise 2.** On Taiwan, train the gradient boosting with `max_iter` of 30, 100, 300 and
1,000 (with `learning_rate=0.05`). Does the test AUC always improve? What would you tell someone
who proposes "more rounds"?

**Exercise 3.** Train the Taiwan random forest with `max_features` equal to 1, 4 (close to
the square root of 23) and 23 (all the variables, which is pure bagging). Which gives the best AUC?
Relate it to $\rho$ in the formula for the variance of the average.


## Solutions


In [ ]:

# SOLUTION 1 -- more trees in the bagging by hand
rng = np.random.default_rng(11); B25 = 25
muestras25 = [rng.integers(0, n, n) for _ in range(B25)]
arboles25 = [DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train.iloc[m], y_train.iloc[m]) for m in muestras25]
votos25 = np.column_stack([a.predict(X_test) for a in arboles25]); mayoria25 = (votos25.mean(axis=1) > 0.5).astype(int)
cuentas25 = np.column_stack([np.bincount(m, minlength=n) for m in muestras25])
oob = []; sin_arbol = 0
for j in range(n):
    fuera = [b for b in range(B25) if cuentas25[j, b] == 0]
    if not fuera: sin_arbol += 1; continue
    oob.append(int(np.mean([arboles25[b].predict(X_train.iloc[[j]])[0] for b in fuera]) > 0.5) == y_train.iloc[j])
print(f"B = 25: vote on test {accuracy_score(y_test, mayoria25):.3f}   out-of-bag {np.mean(oob):.3f}   firms without any tree that did not see them: {sin_arbol}")
print("With 25 trees, each firm is left out of about 9: the OOB stabilizes. The vote on test changes little: the limit is the information in the data, not the number of trees.")


In [ ]:

# SOLUTION 2 -- how many rounds
for it in [30, 100, 300, 1000]:
    t0 = time.time()
    m = HistGradientBoostingClassifier(max_iter=it, learning_rate=0.05, max_depth=4, early_stopping=False, random_state=0).fit(Xt_train, yt_train)
    print(f"{it:5d} rounds: test AUC {roc_auc_score(yt_test, m.predict_proba(Xt_test)[:, 1]):.4f}   ({time.time() - t0:.0f} s)")
print("The AUC rises, plateaus and ends up falling: more rounds means more overfitting. It is chosen with early stopping on validation, not 'as many as possible'.")


In [ ]:

# SOLUTION 3 -- how many variables per split
for mf in [1, 4, 23]:
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features=mf, n_jobs=-1, random_state=0).fit(Xt_train, yt_train)
    print(f"max_features = {mf:2d}: test AUC {roc_auc_score(yt_test, m.predict_proba(Xt_test)[:, 1]):.4f}")
print("With all the variables (pure bagging) the trees look alike (high rho) and averaging helps less: it is the worst of the three. With 1 or with 4 the trees look less alike and the forest improves; here the two options tie.")
